# PASCAL VOC 2012 — Efficient Semantic Segmentation

This notebook trains a lightweight segmentation model (EfficientSegNet with depthwise separable convolutions) on PASCAL VOC 2012, then evaluates Dice score and FLOPs (efficiency).

## 1. Install Dependencies

Install: `albumentations` (augmentation), `segmentation-models-pytorch`, `fvcore` (FLOPs), `opencv-python`.

In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
# Install required packages: albumentations (augmentation),
# segmentation-models-pytorch, fvcore (FLOPs), opencv-python

!pip install albumentations
!pip install segmentation-models-pytorch
!pip install fvcore
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=9877eb0e1dcad68b95da0a10d8c4b14651a2efa0a29a7076f991328bb4272766
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=374acf3e65395f5d0e2f19adab8a407a24a360ab8bb41de914c074c36315d094
  Stored in directory: /root/.cache/pip/wheels/7c/96/04/4f5f31ff812f684f69f40cb1634357812220aac58d4698048c
Successfully built fvcore iopath


## 2. Download and Extract VOC 2012 Dataset

Download the PASCAL VOC 2012 train/val archive and extract it.

In [2]:
# ============================================================
# CELL 2: Download and Extract VOC 2012 Dataset
# ============================================================
# Download PASCAL VOC 2012 train/val archive and extract it

!wget http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
!tar -xf VOCtrainval_11-May-2012.tar

--2026-03-15 16:32:51--  http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
Resolving host.robots.ox.ac.uk (host.robots.ox.ac.uk)... 129.67.94.50
Connecting to host.robots.ox.ac.uk (host.robots.ox.ac.uk)|129.67.94.50|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar [following]
--2026-03-15 16:32:51--  https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar [following]
--2026-03-15 16:32:52--  https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
Resolving thor.robots.ox.ac.uk (thor.ro

## 3. Imports

Libraries: PyTorch, OpenCV, albumentations, sklearn, fvcore, tqdm.

In [3]:
# ============================================================
# CELL 3: Imports
# ============================================================
# Standard libs, PyTorch, albumentations, segmentation_models_pytorch,
# fvcore (FLOPs), tqdm

import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from fvcore.nn import FlopCountAnalysis
from tqdm import tqdm

## 4. Set Compute Device

Use CUDA if available, otherwise CPU.

In [4]:
# ============================================================
# CELL 4: Set Compute Device (CUDA / CPU)
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## 5. Dataset Paths and Train/Validation Split

Set paths for images, masks, and splits; split train IDs into 80% train / 20% validation.

In [5]:
# ============================================================
# CELL 5: Dataset Paths and Train/Validation Split
# ============================================================
# Set VOC2012 paths (images, masks, splits), load train IDs,
# split into 80% train / 20% validation

base_path = "VOCdevkit/VOC2012"

image_dir = os.path.join(base_path,"JPEGImages")
mask_dir = os.path.join(base_path,"SegmentationClass")

split_dir = os.path.join(base_path,"ImageSets/Segmentation")

with open(os.path.join(split_dir,"train.txt")) as f:
    train_ids = f.read().splitlines()

images = [img + ".jpg" for img in train_ids]

train_imgs, val_imgs = train_test_split(
    images,
    test_size=0.2,
    random_state=42
)

print(len(train_imgs), len(val_imgs))

1171 293


## 6. Data Augmentation (Train & Validation Transforms)

Train: resize 300×300, horizontal flip, Gaussian noise, motion blur, brightness/contrast; ImageNet normalization. Validation: resize and normalize only.

In [6]:
# ============================================================
# CELL 6: Data Augmentation (Train & Validation Transforms)
# ============================================================
# Train: resize 300x300, horizontal flip, Gauss noise, motion blur,
# brightness/contrast; ImageNet normalization. Val: resize + normalize only.

train_transform = A.Compose([
    A.Resize(300,300),

    A.HorizontalFlip(p=0.5),

    A.GaussNoise(p=0.3),

    A.MotionBlur(blur_limit=3,p=0.2),

    A.RandomBrightnessContrast(p=0.2),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(300,300),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

## 7. VOCDataset Class (PyTorch Dataset)

Loads image and mask from paths, applies transform; mask values 255→0 and clipped to 0–20 (21 classes).

In [6]:
# ============================================================
# CELL 7: (Reserved / Placeholder)
# ============================================================
# Optional: add extra setup or experiments here.

## 8. Create DataLoaders

Build train and validation DataLoaders (batch_size=8, 2 workers).

In [7]:
# ============================================================
# CELL 8: VOCDataset Class (PyTorch Dataset)
# ============================================================
# Loads image + mask from paths, applies transform; masks 255→0, clipped to 0–20 (21 classes).

class VOCDataset(Dataset):

    def __init__(self, img_list, transform):
        self.img_list = img_list
        self.transform = transform

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):

        img_name = self.img_list[idx]

        img_path = os.path.join(image_dir,img_name)
        mask_path = os.path.join(mask_dir,img_name.replace(".jpg",".png"))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path,0)

        mask[mask==255] = 0
        mask = np.clip(mask,0,20)

        augmented = self.transform(image=image,mask=mask)

        image = augmented["image"]
        mask = augmented["mask"]

        return image.float(), mask.long()

## 9. DSConv — Depthwise Separable Convolution

Depthwise 3×3 conv + pointwise 1×1 conv + BatchNorm + ReLU; used for efficient segmentation.

In [8]:
# ============================================================
# CELL 9: Create DataLoaders (Train & Validation)
# ============================================================
# Build VOCDataset instances and DataLoaders (batch_size=8, 2 workers)

train_dataset = VOCDataset(train_imgs,train_transform)
val_dataset = VOCDataset(val_imgs,val_transform)

train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True,num_workers=2)
val_loader = DataLoader(val_dataset,batch_size=8,num_workers=2)

## 10. EfficientSegNet — Custom Segmentation Model

Encoder (3 DSConv stages) → bottleneck → decoder (bilinear upsample + DSConv) → 1×1 head; 21 classes.

In [9]:
# ============================================================
# CELL 10: DSConv — Depthwise Separable Convolution Module
# ============================================================
# Depthwise conv (3x3) + pointwise conv (1x1) + BN + ReLU; used for efficient segmentation.

import torch.nn as nn

class DSConv(nn.Module):

    def __init__(self,in_c,out_c,stride=1):
        super().__init__()

        self.depth = nn.Conv2d(
            in_c,
            in_c,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_c,
            bias=False
        )

        self.point = nn.Conv2d(
            in_c,
            out_c,
            kernel_size=1,
            bias=False
        )

        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)

    def forward(self,x):

        x = self.depth(x)
        x = self.point(x)
        x = self.bn(x)

        return self.relu(x)

## 11. Instantiate Model

Create the model and move it to the selected device.

In [10]:
# ============================================================
# CELL 11: EfficientSegNet — Custom Segmentation Model
# ============================================================
# Encoder (3 DSConv stages) → bottleneck → decoder (bilinear upsample + DSConv) → 1x1 head; 21 classes.

class EfficientSegNet(nn.Module):

    def __init__(self,num_classes=21):
        super().__init__()

        self.enc1 = DSConv(3,20,stride=2)
        self.enc2 = DSConv(20,40,stride=2)
        self.enc3 = DSConv(40,64,stride=2)

        self.bottleneck = DSConv(64,64)

        self.dec3 = DSConv(64,40)
        self.dec2 = DSConv(40,20)
        self.dec1 = DSConv(20,20)

        self.head = nn.Conv2d(20,num_classes,1)


    def forward(self,x):

        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)

        x = self.bottleneck(x)

        x = F.interpolate(x,scale_factor=2,mode="bilinear",align_corners=False)
        x = self.dec3(x)

        x = F.interpolate(x,scale_factor=2,mode="bilinear",align_corners=False)
        x = self.dec2(x)

        x = F.interpolate(x,scale_factor=2,mode="bilinear",align_corners=False)
        x = self.dec1(x)

        x = F.interpolate(x,size=(300,300),mode="bilinear",align_corners=False)

        return self.head(x)

## 12. Loss Functions

Cross-entropy for classification; Dice loss (1 − mean per-class Dice) with softmax and one-hot targets.

In [11]:
# ============================================================
# CELL 12: Instantiate Model and Move to Device
# ============================================================

model = EfficientSegNet(num_classes=21).to(device)

## 13. Optimizer and Hyperparameters

AdamW optimizer (lr=1e-3), 30 epochs.

In [12]:
# ============================================================
# CELL 13: Loss Functions (Cross-Entropy + Dice)
# ============================================================
# CE for classification; dice_loss = 1 - mean per-class Dice (softmax + one-hot).

ce_loss = nn.CrossEntropyLoss()

def dice_loss(pred,target):

    pred = torch.softmax(pred,dim=1)

    target_one_hot = torch.nn.functional.one_hot(
        target,21).permute(0,3,1,2).float()

    intersection = (pred*target_one_hot).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target_one_hot.sum(dim=(2,3))

    dice = (2*intersection+1e-6)/(union+1e-6)

    return 1-dice.mean()

## 14. compute_dice — Validation Metric

Mean per-class Dice over 21 classes (argmax predictions).

In [13]:
# ============================================================
# CELL 14: Optimizer and Training Hyperparameters
# ============================================================
# AdamW optimizer (lr=1e-3), 30 epochs

optimizer = optim.AdamW(model.parameters(),lr=1e-3)

epochs = 30

## 15. Training Loop

Train for 30 epochs with loss = CE + 2×Dice; validate each epoch; save `best_model.pth` when validation Dice improves.

In [14]:
# ============================================================
# CELL 15: compute_dice — Validation Metric (Mean Per-Class Dice)
# ============================================================
# Argmax predictions; for each of 21 classes compute Dice; return mean.

def compute_dice(pred,target):

    pred = torch.argmax(pred,dim=1)

    scores=[]

    for c in range(21):

        p = pred==c
        t = target==c

        inter = (p&t).sum().float()
        union = p.sum()+t.sum()

        if union==0:
            scores.append(torch.tensor(1.0,device=device))
        else:
            scores.append((2*inter)/(union+1e-6))

    return torch.mean(torch.stack(scores))

## 16. Compute Model FLOPs (GFLOPs)

Single forward pass on 1×3×300×300 input; report total FLOPs in GFLOPs using fvcore.

In [15]:
# ============================================================
# CELL 16: Training Loop (CE + 2*Dice Loss, Save Best by Val Dice)
# ============================================================
# Train for `epochs`; each step: CE + 2*dice_loss; validate; save best_model.pth if val Dice improves.

best_dice = 0

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for img,mask in tqdm(train_loader):

        img = img.to(device)
        mask = mask.to(device)

        pred = model(img)

        loss = ce_loss(pred,mask) + 2*dice_loss(pred,mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_loss = train_loss/len(train_loader)

    model.eval()
    dice_scores=[]

    with torch.no_grad():

        for img,mask in val_loader:

            img = img.to(device)
            mask = mask.to(device)

            pred = model(img)

            dice = compute_dice(pred,mask)

            dice_scores.append(dice.item())

    avg_dice = np.mean(dice_scores)

    if avg_dice>best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(),"best_model.pth")

    print(f"Epoch {epoch+1}")
    print("Train Loss:",avg_loss)
    print("Val Dice:",avg_dice)

100%|██████████| 147/147 [00:13<00:00, 10.93it/s]


Epoch 1
Train Loss: 3.7812737137281975
Val Dice: 0.9171511643641704


100%|██████████| 147/147 [00:11<00:00, 12.74it/s]


Epoch 2
Train Loss: 2.705764558039555
Val Dice: 0.9192464963809864


100%|██████████| 147/147 [00:11<00:00, 12.71it/s]


...

Epoch 28
Train Loss: 2.4837231344106248
Val Dice: 0.9241855643890999


100%|██████████| 147/147 [00:11<00:00, 12.41it/s]


Epoch 29
Train Loss: 2.483350611057411
Val Dice: 0.9240429256413434


100%|██████████| 147/147 [00:11<00:00, 12.40it/s]


Epoch 30
Train Loss: 2.482630744272349
Val Dice: 0.9245356369662929


## 17. Final Validation — Dice and Dice/FLOPs Ratio

Run on validation set; report mean Dice and Dice/GFLOPs (efficiency metric).

In [16]:
# ============================================================
# CELL 17: Compute Model FLOPs (GFLOPs) with fvcore
# ============================================================
# Single forward pass on 1x3x300x300 input; report total FLOPs in GFLOPs.

model.eval()

dummy = torch.randn(1,3,300,300).to(device)

flops = FlopCountAnalysis(model,dummy)

gflops = flops.total()/1e9

print("GFLOPs:",gflops)

GFLOPs: 0.181319424


## 18. ZIP Dataset Evaluation

Prompt for a zip path; extract; find `JPEGImages` and `SegmentationClass`; load `best_model.pth`; compute Dice, GFLOPs, parameters, and Dice/FLOPs; print summary table.

In [17]:
# ============================================================
# CELL 18: Final Validation — Mean Dice and Dice/FLOPs Ratio
# ============================================================
# Run model on validation set; report mean Dice and Dice/GFLOPs (efficiency metric).

model.eval()

dice_scores=[]

with torch.no_grad():

    for img,mask in val_loader:

        img = img.to(device)
        mask = mask.to(device)

        pred = model(img)

        dice = compute_dice(pred,mask)

        dice_scores.append(dice.item())

avg_dice = np.mean(dice_scores)

ratio = avg_dice/gflops

print("Dice:",avg_dice)
print("Dice/FLOPs:",ratio)

Dice: 0.9245356369662929
Dice/FLOPs: 5.098933233795695


In [18]:
# ============================================================
# CELL 19: ZIP Dataset Evaluation — Load Zip, Run Model, Report Metrics
# ============================================================
# Prompts for zip path; extracts; finds JPEGImages/SegmentationClass;
# builds TestDataset/DataLoader; loads best_model.pth; computes Dice,
# GFLOPs, params, Dice/FLOPs; prints summary table.

# =========================================
# ZIP DATASET EVALUATION CELL
# =========================================

import os
import zipfile
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from fvcore.nn import FlopCountAnalysis

# -----------------------------------------
# Device
# -----------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

# -----------------------------------------
# Ask for ZIP file
# -----------------------------------------

zip_path = input("/content/augmented_test_sample.zip")

extract_path = "evaluation_dataset"

with zipfile.ZipFile(zip_path,'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:",extract_path)

# -----------------------------------------
# Find dataset folders automatically
# -----------------------------------------

IMG_DIR = None
MASK_DIR = None

for root,dirs,files in os.walk(extract_path):

    if "JPEGImages" in dirs:
        IMG_DIR = os.path.join(root,"JPEGImages")

    if "SegmentationClass" in dirs:
        MASK_DIR = os.path.join(root,"SegmentationClass")

print("Images folder:",IMG_DIR)
print("Masks folder:",MASK_DIR)

# -----------------------------------------
# Transform
# -----------------------------------------

transform = A.Compose([
    A.Resize(300,300),
    ToTensorV2()
])

# -----------------------------------------
# Dataset
# -----------------------------------------

class TestDataset(Dataset):

    def __init__(self,img_dir,mask_dir,transform):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = sorted(os.listdir(img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self,idx):

        name = self.files[idx]

        img = Image.open(os.path.join(self.img_dir,name)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir,name.replace(".jpg",".png")))

        img = np.array(img)
        mask = np.array(mask)

        aug = self.transform(image=img,mask=mask)

        return aug["image"].float(), aug["mask"].long()

dataset = TestDataset(IMG_DIR,MASK_DIR,transform)
loader = DataLoader(dataset,batch_size=8,shuffle=False)

print("Test samples:",len(dataset))

# -----------------------------------------
# Dice metric
# -----------------------------------------

def compute_dice(pred,target):

    pred = torch.argmax(pred,dim=1)

    mask = target!=255
    pred = pred[mask]
    target = target[mask]

    scores=[]

    for c in range(21):

        p = pred==c
        t = target==c

        inter = (p&t).sum().float()
        union = p.sum()+t.sum()

        if union==0:
            scores.append(torch.tensor(1.0,device=device))
        else:
            scores.append((2*inter)/(union+1e-6))

    return torch.mean(torch.stack(scores))

# -----------------------------------------
# Load model
# -----------------------------------------

model = EfficientSegNet(num_classes=21).to(device)

model.load_state_dict(torch.load("best_model.pth",map_location=device))

model.eval()

print("Model loaded")

# -----------------------------------------
# Evaluation
# -----------------------------------------

dice_score = 0

with torch.no_grad():

    for images,masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        pred = model(images)

        dice_score += compute_dice(pred,masks).item()

dice_score /= len(loader)

print("\nDice Score:",round(dice_score,4))

# -----------------------------------------
# FLOPs
# -----------------------------------------

dummy = torch.randn(1,3,300,300).to(device)

flops = FlopCountAnalysis(model,dummy)

gflops = flops.total()/1e9

params = sum(p.numel() for p in model.parameters())

ratio = dice_score/gflops

# -----------------------------------------
# Summary
# -----------------------------------------

data = {
    "Metric":[
        "Dice Score",
        "FLOPs (GFLOPs)",
        "Parameters",
        "Dice/FLOPs Ratio"
    ],
    "Value":[
        round(dice_score,4),
        round(gflops,4),
        params,
        round(ratio,4)
    ]
}

df = pd.DataFrame(data)

print("\nMODEL PERFORMANCE SUMMARY\n")
print(df.to_string(index=False))

Device: cuda
/content/augmented_test_sample.zip/content/augmented_test_sample.zip
Dataset extracted to: evaluation_dataset
Images folder: evaluation_dataset/JPEGImages
Masks folder: evaluation_dataset/SegmentationClass
Test samples: 100
Model loaded

Dice Score: 0.4961

MODEL PERFORMANCE SUMMARY

          Metric      Value
      Dice Score     0.4961
  FLOPs (GFLOPs)     0.1813
      Parameters 14512.0000
Dice/FLOPs Ratio     2.7363
